In [20]:
import os
import shutil
import pandas as pd

In [21]:
FPS = 15
DATASET = "tracking/val_utils/data/MOT17"
NEW_DATASET = DATASET + f'{FPS}FPS'
shutil.copytree(DATASET, NEW_DATASET, dirs_exist_ok=True)

'tracking/val_utils/data/MOT1715FPS'

In [22]:
sequences = os.listdir(f"{NEW_DATASET}/train")
for seq in sorted(sequences):
    print(seq)
    cur_fps = int(open(f"{NEW_DATASET}/train/{seq}/seqinfo.ini").readlines()[3].split('=')[-1])
    cur_len = int(open(f"{NEW_DATASET}/train/{seq}/seqinfo.ini").readlines()[4].split('=')[-1])
    RM_FPS = max(1, cur_fps // FPS)
    print(cur_fps, RM_FPS)
    # GT clean
    df = pd.read_csv(f"{NEW_DATASET}/train/{seq}/gt/gt.txt", names=[str(i) for i in range(9)], header=None)
    df = df[df['0'] % RM_FPS == 0].copy()
    df['0'] = df['0'] // RM_FPS
    df.to_csv(f"{NEW_DATASET}/train/{seq}/gt/gt.txt", index=False, header=False)
    # Img clean
    directory = f"{NEW_DATASET}/train/{seq}/img1/"
    a = os.listdir(directory)
    a.sort()
    for filename in a:
        if filename.endswith('.jpg'):
            # Extract the numeric part from the file name
            numeric_part = filename.split('.')[0]
            try:
                value = int(numeric_part)
            except ValueError:
                # Skip files that don't have a numeric name
                continue

            # Check if the numeric value is divisible by X
            if value % RM_FPS != 0:
                # Remove the file if condition is not met
                os.remove(os.path.join(directory, filename))
            else:
                # Calculate the new name using integer division
                new_value = value // RM_FPS
                # Format the new number with leading zeros (6 digits, as in original names)
                new_filename = f"{new_value:06d}.jpg"
                # Rename the file
                os.rename(os.path.join(directory, filename), os.path.join(directory, new_filename))

MOT17-02
30 2
MOT17-04
30 2
MOT17-05
14 1
MOT17-09
30 2
MOT17-10
30 2
MOT17-11
30 2
MOT17-13
25 1
